In [19]:
import os
import pandas as pd

directory_path = 'ncaa_box_scores_2024_25'

dataframes = []

for file_name in os.listdir(directory_path):
    file_path = os.path.join(directory_path, file_name)
    df = pd.read_csv(file_path)
    dataframes.append(df)

final_dataframe = pd.concat(dataframes, ignore_index=True)

final_dataframe['team_score'] = final_dataframe['team_score'] / 2
final_dataframe['opponent_score'] = final_dataframe['opponent_score'] / 2

final_dataframe

,player_id,player,MIN,FGM,FGA,3PTM,3PTA,FTM,FTA,OREB,...,team,opponent,home,starter,game_id,date,location,team_score,opponent_score,result
0,5177717.0,M. Gomma,26.0,3,4,0,0,1,2,1,...,Seattle U,Eastern Washington,False,True,401718465,2024-11-06,A,86.0,93.0,L
1,4610395.0,M. Moncrieffe,36.0,7,9,0,0,6,10,3,...,Seattle U,Eastern Washington,False,True,401718465,2024-11-06,A,86.0,93.0,L
2,5241342.0,B. Maldonado,14.0,2,9,2,5,1,1,0,...,Seattle U,Eastern Washington,False,True,401718465,2024-11-06,A,86.0,93.0,L
3,5107830.0,M. Arington,31.0,3,7,1,1,1,1,0,...,Seattle U,Eastern Washington,False,True,401718465,2024-11-06,A,86.0,93.0,L
4,4433606.0,J. Christofilis,31.0,4,11,4,6,5,6,0,...,Seattle U,Eastern Washington,False,True,401718465,2024-11-06,A,86.0,93.0,L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131998,5106735.0,J. Karasinski,15.0,2,6,1,3,0,0,0,...,Bellarmine,Florida Gulf Coast,False,False,401711649,2025-02-26,A,61.0,80.0,L
131999,5239765.0,M. Watkins,23.0,6,10,1,3,0,0,0,...,Bellarmine,Florida Gulf Coast,False,False,401711649,2025-02-26,A,61.0,80.0,L
132000,5175849.0,AJ Lux,23.0,3,5,2,4,0,0,1,...,Bellarmine,Florida Gulf Coast,False,False,401711649,2025-02-26,A,61.0,80.0,L
132001,4702669.0,D. McKinney,7.0,2,3,1,2,2,2,0,...,Bellarmine,Florida Gulf Coast,False,False,401711649,2025-02-26,A,61.0,80.0,L


In [14]:
import pandas as pd

# Drop duplicate game_id rows, keeping only the first occurrence
unique_games = final_dataframe.drop_duplicates(subset=['game_id'])

# Create a new DataFrame with the required columns
filtered_df = pd.DataFrame({
    'game_date': unique_games['date'],
    'game_id': unique_games['game_id'],
    'team_a': unique_games['team'],  # Assuming 'team_name' represents team A
    'team_b': unique_games['opponent'],  # Assuming 'opponent_name' represents team B
    'team_a_score': unique_games['team_score'],
    'team_b_score': unique_games['opponent_score'],
    'team_a_location': unique_games['location'],  # Assuming 'location' represents team A's location
    'team_a_lost': unique_games['team_score'] < unique_games['opponent_score']
})

filtered_df

,game_date,game_id,team_a,team_b,team_a_score,team_b_score,team_a_location,team_a_lost
0,2024-11-06,401718465,Seattle U,Eastern Washington,86.0,93.0,A,True
11,2024-11-09,401718707,Seattle U,Liberty,64.0,66.0,H,True
23,2024-11-14,401725507,Seattle U,Cal Poly,71.0,75.0,A,True
35,2024-11-16,401722415,Seattle U,UC San Diego,84.0,71.0,A,False
46,2024-11-22,401725508,Seattle U,Northwest University,80.0,52.0,H,False
...,...,...,...,...,...,...,...,...
130951,2024-11-07,401732252,North Texas,Wayland Baptist,80.0,38.0,H,False
130987,2024-11-21,401734106,North Texas,Texas Wesleyan,73.0,66.0,H,False
131436,2024-12-18,401727288,Green Bay,Michigan Tech,70.0,72.0,H,True
131713,2024-11-26,401727263,Bellarmine,Midway University,101.0,60.0,H,False


In [15]:
# Ensure the game_date column is in datetime format
filtered_df['game_date'] = pd.to_datetime(filtered_df['game_date'])

# Sort the DataFrame by game_date in ascending order
filtered_df = filtered_df.sort_values(by='game_date', ascending=True)

filtered_df

,game_date,game_id,team_a,team_b,team_a_score,team_b_score,team_a_location,team_a_lost
22087,2024-11-04,401721017,South Alabama,Central Michigan,70.0,74.0,H,True
63621,2024-11-04,401721028,Kent State,Louisiana,70.0,66.0,A,False
64392,2024-11-04,401707875,Cleveland State,Michigan,53.0,101.0,A,True
9051,2024-11-04,401727413,Western Carolina,Bob Jones,100.0,61.0,H,False
110012,2024-11-04,401726405,New Mexico State,New Mexico Highlands,93.0,57.0,H,False
...,...,...,...,...,...,...,...,...
82042,2025-04-05,401758632,Nebraska,Boise State,79.0,69.0,N,False
101614,2025-04-05,401746081,Florida,Auburn,79.0,73.0,N,False
16722,2025-04-05,401746080,Houston,Duke,70.0,67.0,N,False
27594,2025-04-06,401758636,UCF,Nebraska,66.0,77.0,N,True


In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Ensure the game_date column is in datetime format
filtered_df['game_date'] = pd.to_datetime(filtered_df['game_date'])

# Split the data into training and testing sets
train_data = filtered_df[filtered_df['game_date'] <= '2025-02-28']
test_data = filtered_df[(filtered_df['game_date'] >= '2025-03-01') & (filtered_df['game_date'] <= '2025-04-07')]

# Define features and target for training
train_features = ['team_a_location']  # Use available features other than scores
target = 'team_a_lost'

X_train = train_data[train_features]
y_train = train_data[target]

# Define features for testing (exclude scores)
test_features = ['team_a_location']
X_test = test_data[test_features]
y_test = test_data[target]

# Train a Random Forest Classifier
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy:.2f}")

# Iterate through the test dataset and print results for each game
for index, row in test_data.iterrows():
    game_id = row['game_id']
    team_a = row['team_a']
    team_b = row['team_b']
    actual_result = 'Lost' if row['team_a_lost'] else 'Won'
    predicted_result = 'Lost' if model.predict([[row['team_a_location']]])[0] else 'Won'

    print(f"Game ID: {game_id}, {team_a} vs {team_b}")
    print(f"Actual Result: Team A {actual_result}, Predicted Result: Team A {predicted_result}")
    print("-" * 50)

ValueError: could not convert string to float: 'H'